In [1]:
code1 = """
#include <stdio.h>
#include <cuda_runtime.h>
#include <math.h>

// Kernel to compute SUM of elements
__global__ void ComputeSum(float *data, int N, float *sum) {
    float partial = 0.0f;
    for (int i = threadIdx.x; i < N; i += blockDim.x) {
        partial += data[i];
        printf("[Kernel SUM] Thread %d partial = %f\\n", threadIdx.x, partial);
    }
    atomicAdd(sum, partial);
    printf("\\n[Kernel SUM] Thread %d sum = %f\\n", threadIdx.x, *sum);
}

// Kernel to compute SUM of squared diffs from mean
__global__ void ComputeStdDev(float *data, int N, float *mean, float *sumsq) {
    float partial = 0.0f;
    float m = *mean;
    for (int i = threadIdx.x; i < N; i += blockDim.x) {
        float diff = data[i] - m;
        partial += diff * diff;
        printf("[Kernel StdDev] Thread %d partial = %f\\n", threadIdx.x, partial);
    }
    atomicAdd(sumsq, partial);
    printf("[Kernel StdDev] Thread %d sumsq = %f\\n\\n", threadIdx.x, *sumsq);
}

int main() {
    const int N = 16;
    float data[N] = {1.0, 2.0, 3.0, 4.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0};

    // Device memory
    float *d_data, *d_sum, *d_sumsq, *d_mean;
    cudaMalloc(&d_data, N * sizeof(float));
    cudaMalloc(&d_sum, sizeof(float));
    cudaMalloc(&d_sumsq, sizeof(float));
    cudaMalloc(&d_mean, sizeof(float));

    // Create streams and event
    cudaStream_t stream1, stream2;
    cudaStreamCreate(&stream1);
    cudaStreamCreate(&stream2);
    cudaEvent_t meanComputed;
    cudaEventCreate(&meanComputed);

    // Copy input
    cudaMemcpyAsync(d_data, data, N * sizeof(float), cudaMemcpyHostToDevice, stream1);
    cudaMemsetAsync(d_sum, 0, sizeof(float), stream1);
    cudaMemsetAsync(d_sumsq, 0, sizeof(float), stream2);

    // --- Step 1: Compute SUM (on stream1) ---
    ComputeSum<<<1, 4, 0, stream1>>>(d_data, N, d_sum);
    cudaEventRecord(meanComputed, stream1);   // mark completion of ComputeSum

    // Wait for meanComputed before launching Step 2
    cudaStreamWaitEvent(stream2, meanComputed, 0);

    // Copy sum back asynchronously
    float sum;
    cudaMemcpyAsync(&sum, d_sum, sizeof(float), cudaMemcpyDeviceToHost, stream1);

    // Synchronize stream1 so we know sum is ready
    //cudaStreamSynchronize(stream1);

    // Compute mean on host
    float mean = sum / N;
    cudaMemcpyAsync(d_mean, &mean, sizeof(float), cudaMemcpyHostToDevice, stream2);

    // --- Step 2: Compute squared diffs (on stream2) ---
    ComputeStdDev<<<1, 4, 0, stream2>>>(d_data, N, d_mean, d_sumsq);

    // Copy results back
    float sumsq, stddev;
    cudaMemcpyAsync(&sumsq, d_sumsq, sizeof(float), cudaMemcpyDeviceToHost, stream2);

    // Synchronize stream2 (wait for all GPU work to finish)
    cudaStreamSynchronize(stream2);

    stddev = sqrt(sumsq / N);

    // Final result
    printf("mean = %f, stddev = %f\\n", mean, stddev);

    // Cleanup
    cudaFree(d_data);
    cudaFree(d_sum);
    cudaFree(d_sumsq);
    cudaFree(d_mean);
    cudaStreamDestroy(stream1);
    cudaStreamDestroy(stream2);
    cudaEventDestroy(meanComputed);

    return 0;
}

"""


with open("/tmp/stdDeviation1.cu", "w") as f:
    f.write(code1)

In [ ]:
4

In [ ]:
!./stdDeviation_debug

[Kernel SUM] Thread 0 partial = 1.000000
[Kernel SUM] Thread 1 partial = 2.000000
[Kernel SUM] Thread 2 partial = 3.000000
[Kernel SUM] Thread 3 partial = 4.000000
[Kernel SUM] Thread 0 partial = 7.000000
[Kernel SUM] Thread 1 partial = 9.000000
[Kernel SUM] Thread 2 partial = 11.000000
[Kernel SUM] Thread 3 partial = 13.000000
[Kernel SUM] Thread 0 partial = 17.000000
[Kernel SUM] Thread 1 partial = 20.000000
[Kernel SUM] Thread 2 partial = 23.000000
[Kernel SUM] Thread 3 partial = 26.000000
[Kernel SUM] Thread 0 partial = 31.000000
[Kernel SUM] Thread 1 partial = 35.000000
[Kernel SUM] Thread 2 partial = 39.000000
[Kernel SUM] Thread 3 partial = 26.000000

[Kernel SUM] Thread 0 sum = 131.000000

[Kernel SUM] Thread 1 sum = 131.000000

[Kernel SUM] Thread 2 sum = 131.000000

[Kernel SUM] Thread 3 sum = 131.000000
[Kernel StdDev] Thread 0 partial = 51.660156
[Kernel StdDev] Thread 1 partial = 38.285156
[Kernel StdDev] Thread 2 partial = 26.910156
[Kernel StdDev] Thread 3 partial = 17.5